## A simple CNN in Pytorch
Following [Michael Li](https://towardsdatascience.com/build-a-fashion-mnist-cnn-pytorch-style-efb297e22582), who explains how to do all of this. See also the Pytorch tutorials, where Li probably got his information.

In [1]:
# Plotting and numeric libraries
import torch
import bokeh
from bokeh.plotting import figure, output_notebook, show, output_file
from bokeh.models import Label
import numpy as np
import pandas as pd
import tqdm.auto as tqdm
import tarfile

# sklearn utilities
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder

print(f"""Using Torch version {torch.__version__}.  
        CUDA is {'available' if torch.cuda.is_available() else 'not available'}. 
        MPS is {'available' if torch.backends.mps.is_available() else 'not available'}""")
gpu = 'mps' if torch.backends.mps.is_available() else 'cuda'
cpu = 'cpu'

print(f"Using bokeh version {bokeh.__version__}.")

print(f"Using pandas version {pd.__version__}.")

C:\Users\Luis Mendez\AppData\Roaming\Python\Python312\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
C:\Users\Luis Mendez\AppData\Roaming\Python\Python312\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


Using Torch version 2.5.1+cu121.  
        CUDA is available. 
        MPS is not available
Using bokeh version 3.6.0.
Using pandas version 3.0.1.


In [2]:
output_notebook()
device = gpu

Loading BokehJS ...

In [3]:
# Build the neural network, expand on top of nn.Module
class Network(torch.nn.Module):
    def __init__(self):
        super().__init__()

        # define layers
        self.conv1 = torch.nn.Conv2d(in_channels=1, out_channels=6, kernel_size=5)
        ## takes a 28 x 28 matrix and produces 6 x 24 x 24 matrices
        self.conv2 = torch.nn.Conv2d(in_channels=6, out_channels=12, kernel_size=5)
        ## takes 6 24 x 24 matrices and producs 12 x 20 x 20 matrices

        self.fc1 = torch.nn.Linear(in_features=12*4*4, out_features=120)
        self.fc2 = torch.nn.Linear(in_features=120, out_features=60)
        self.out = torch.nn.Linear(in_features=60, out_features=10)

    # define forward function
    def forward(self, t):
        # conv 1
        t = self.conv1(t)
        ## 1 x 28 x 28 goes to 6 x 24 x 24
        t = torch.nn.functional.relu(t)
        t = torch.nn.functional.max_pool2d(t, kernel_size=2, stride=2)
        ## 6 x 24 x 24 goes to 6 x 12 x 12 through pooling

        # conv 2
        t = self.conv2(t)
        ## 6 x 12 x 12 goes to 12 x 8 x 8 
        t = torch.nn.functional.relu(t)
        t = torch.nn.functional.max_pool2d(t, kernel_size=2, stride=2)
        ## pooling 12 x 4 x 4

        # fc1
        t = t.reshape(-1, 12*4*4)
        t = self.fc1(t)
        t = torch.nn.functional.relu(t)
        ## 120 features in to 60 out

        # fc2
        t = self.fc2(t)
        t = torch.nn.functional.relu(t)
        ## 60 in to 10 out

        # output
        t = self.out(t)
        # don't need softmax here since we'll use cross-entropy as activation.

        return t

In [4]:
with tarfile.open("neural.tgz", "r:gz") as tar:
    # adjust the member name to the CSV inside the archive
    with tar.extractfile("fmnist_data.csv") as f:
        train_images = pd.read_csv(f)
    with tar.extractfile("fmnist_labels.csv") as f:
        train_labels = pd.read_csv(f)
        


train_labels = OneHotEncoder().fit_transform(train_labels).toarray()




In [5]:
train_images.shape

(9999, 784)

In [6]:
Xtrain = torch.tensor(train_images.values, dtype=torch.float32, device=device).reshape((train_images.shape[0], 1, 28, 28))
Ytrain = torch.tensor(train_labels, dtype=torch.float32, device=device).reshape((train_labels.shape[0], train_labels.shape[1]))
criterion = torch.nn.functional.cross_entropy

In [7]:
def train(model, Xt, Yt):
    """One step through the training loop"""
    # reset the gradient calculations
    
    # reset the gradient calculations
    optimizer.zero_grad()
        
    predicted = model(Xt)
    
    # compute the loss
    loss = criterion(predicted,Yt)
    # compute the gradients by backward propogation
    loss.backward()       
    # adjust the weights
    optimizer.step()   
    

    
    return loss.item()

In [8]:
def training_loop(model, data, target,threshold=1e-6,max_iter=100000):
    """Run the training loop and return the losses"""
    

    losses = []
    prior_loss=1000000
    for i in tqdm.tqdm(range(max_iter)):
        loss = train(model,data, target)
        losses.append(loss)
        if abs(loss-prior_loss) < threshold:
            break
        prior_loss = loss
        
    return losses
    

In [9]:
def plot_loss(losses):
    """Plot the losses"""
    f=figure(title="Loss over time",x_axis_label="Epoch",y_axis_label="Loss")
    f.line(x=list(range(len(losses))),y=losses)

    
    return f

In [10]:
model = Network().to(device) #gpu used to compute vector arithmetic, 
optimizer = torch.optim.Adam(model.parameters(), lr=0.00001) #instead of adam, we can also use gradient descent, but adam is more efficient and faster to converge.
losses = training_loop(model, Xtrain,Ytrain,threshold=1e-6,max_iter=10000)
show(plot_loss(losses))

  0%|          | 0/10000 [00:00<?, ?it/s]

In [11]:
(torch.argmax(torch.nn.functional.softmax(model(Xtrain),dim=1),dim=1)==torch.argmax(Ytrain,dim=1)).sum().item()/Xtrain.shape[0]

0.9992999299929993

In [12]:
output_file("test.html")
show(plot_loss(losses))

In [13]:
list(model.named_parameters())

[('conv1.weight',
  Parameter containing:
  tensor([[[[ 0.2550,  0.1303,  0.2451,  0.1671, -0.0869],
            [-0.0481, -0.0835,  0.2631, -0.1172,  0.1430],
            [ 0.1669,  0.1774, -0.0524, -0.1250, -0.1262],
            [-0.1948, -0.1391, -0.0275, -0.2455, -0.0011],
            [-0.1628,  0.0577, -0.2724, -0.2281,  0.0182]]],
  
  
          [[[ 0.1998,  0.0227, -0.0361,  0.1300,  0.2211],
            [-0.1257,  0.1628,  0.1206, -0.0817,  0.1261],
            [ 0.0286, -0.1119,  0.1212,  0.0880,  0.2156],
            [-0.1529,  0.0523,  0.1714, -0.0212, -0.0493],
            [ 0.1503, -0.0014, -0.0441, -0.2009,  0.2330]]],
  
  
          [[[ 0.0541, -0.0657, -0.0202, -0.0091,  0.0294],
            [-0.0578,  0.1199, -0.0389, -0.2391,  0.0969],
            [ 0.0026, -0.1433, -0.2157, -0.1663, -0.0553],
            [ 0.1830,  0.0807, -0.0464, -0.1906,  0.0076],
            [ 0.1878,  0.0734,  0.1336,  0.0799, -0.0121]]],
  
  
          [[[ 0.2069, -0.0426,  0.2025,  0.1840, 